## Shared Setup

Run this cell first to install dependencies and configure the OpenAI client for all examples in this chapter.

In [ ]:
!pip install -q openai
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-your-key-here'))
BASE_MODEL = os.getenv('BASE_MODEL', 'gpt-3.5-turbo-0125')

## Applying Supervised Fine-Tuning

Create minimal, "failure-shaped" SFT examples with stable structure and explicit uncertainty handling, then upload them and launch a supervised fine-tuning job.

In [ ]:
import json
from pathlib import Path

BASE_MODEL = BASE_MODEL  # Provided by shared setup cell (or set explicitly)

# Minimal, "failure-shaped" SFT examples: stable structure, explicit uncertainty handling.
train_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a policy-aware support assistant. If context is insufficient, say 'Information not available' and ask a clarifying question. Output: Summary, Next steps, Risks."},
            {"role": "user", "content": "Summarize incident and propose next steps. Context: API returned 503 after deployment. Rollback resolved issue."},
            {"role": "assistant", "content": "Summary\n503s started after deployment; rollback restored service.\n\nNext steps\n- Diff the deployment for config/routing changes\n- Check gateway/load balancer logs around deploy window\n- Re-enable via canary with health checks\n\nRisks\nRe-deploying without isolating the change can reintroduce outage."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a policy-aware support assistant. If context is insufficient, say 'Information not available' and ask a clarifying question. Output: Summary, Next steps, Risks."},
            {"role": "user", "content": "Export customer emails to CSV and send to vendor. Context: none."},
            {"role": "assistant", "content": "Summary\nInformation not available to proceed.\n\nNext steps\n- Confirm business purpose and approval path\n- Confirm data classification and sharing policy\n- Confirm whether a data processing agreement exists\n\nRisks\nSharing customer data without approvals and controls can violate policy and contracts."},
        ]
    },
]

valid_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a policy-aware support assistant. If context is insufficient, say 'Information not available' and ask a clarifying question. Output: Summary, Next steps, Risks."},
            {"role": "user", "content": "Latency spiked after enabling new cache. Disabling cache restored performance. Summarize and propose next steps."},
            {"role": "assistant", "content": "Summary\nLatency increased after enabling the cache; disabling it restored performance.\n\nNext steps\n- Compare cache config (TTL/keying/eviction) before vs after\n- Inspect hit rate and backend saturation during spike\n- Re-enable via canary and isolate the specific setting\n\nRisks\nRe-enabling without isolating the setting can recreate the incident."},
        ]
    }
]

def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

train_path = Path("chapter07_sft_train.jsonl")
valid_path = Path("chapter07_sft_valid.jsonl")
write_jsonl(train_path, train_examples)
write_jsonl(valid_path, valid_examples)

# Upload files for fine-tuning
with train_path.open("rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with valid_path.open("rb") as f:
    valid_file = client.files.create(file=f, purpose="fine-tune")

# Create SFT job (keep hyperparameters default unless you have a reason)
job = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=train_file.id,
    validation_file=valid_file.id,
    method={"type": "supervised"},
)

print("SFT job:", job.id, "| status:", job.status)